In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### variables

In [33]:
np.random.random(45)

STUDENT_ID = "ravishr.mca25@cs.du.ac.in"

N_STEPS = 120
HISTORY_WINDOW = 10
ALPHA = 0.7
BASE_WEIGHTS = {
    "Behaviour" : 0.35,
    "Context" : 0.2,
    "History" : 0.2,
    "Risk" : 0.25
}
THRESHOLDS = {
    "ALLOW" : 0.75,
    "MONITOR" : 0.5,
    "RESTRICT" : 0.25,
}

In [34]:
DEVICES = {
    "D1": dict(
            Device_Type="Smart Camera",
            mean=120., std=15., active=(0, 24),
            allowed=["Cloud server", "NVR"],
            criticality=.6,
            scenario="full",
            attack_start=60
          ),
    "D2": dict(
            Device_Type="Smart Sensor",
            mean=10., std=2., active=(0, 24),
            allowed=["Gateway"],
            criticality=.4,
            scenario="normal",
            attack_start=None
          ),
    "D3": dict(
            Device_Type="Smart Lock",
            mean=2., std=.5, active=(6, 23),
            allowed=["Home hub"],
            criticality=.9,
            scenario="full",
            attack_start=60
          ),
    "D4": dict(
            Device_Type="Smart TV",
            mean=200., std=60., active=(16, 24),
            allowed=["Streaming servers"],
            criticality=.3,
            scenario="suspicious",
            attack_start=40
          ),
    "D5": dict(
            Device_Type="Medical/Environmental Sensor",
            mean=20., std=3., active=(0, 24),
            allowed=["Hospital gateway"],
            criticality=1.,
            scenario="full",
            attack_start=70
          ),
}

In [35]:
def clip01(value):
    return float(np.clip(value, 0.0, 1.0))

### Get cycle state

In [36]:
def get_state(device, t):
    scenario = device["scenario"]

    if scenario == "normal":
        return "Normal"
    if scenario == "suspicious":
        return "Normal" if t < 40 else ("Suspicious" if t < 80 else "Recovery")

    start = device["attack_start"]

    if t < start - 20:
        return "Normal"
    if t < start:
        return "Suspicious"
    if t < start + 20:
        return "Attack-like"
    return "Recovery"

### Get recovery progress value

In [37]:
def recovery_progress(device, t):
    start = device["attack_start"] + 20
    return clip01((t - start + 1) / 20.0)

### Generate observation for a device at time t

In [38]:
def generate_observation(device, t, state, rng):
    mean, std = device["mean"], device["std"]
    hour = t % 24
    allowed = device["allowed"]

    if state == "Normal":
        rate = rng.normal(mean, std)
        failed = int(rng.choice([0, 1], p=[.95, .05]))
        destination = rng.choice(allowed)
    elif state == "Suspicious":
        rate = rng.normal(mean * rng.uniform(1.5, 2.0), std * 1.5)
        failed = int(rng.integers(0, 3))
        destination = rng.choice(allowed) if rng.random() < .70 else "Unknown host"
    elif state == "Attack-like":
        rate = rng.normal(mean * rng.uniform(4.0, 8.0), max(std, mean * .5))
        failed = int(rng.integers(3, 8))
        destination = "Unknown host" if rng.random() < .85 else rng.choice(allowed)
    else:
        progress = recovery_progress(device, t)
        blended_mean = mean * 2.0 * (1-progress) + mean * progress
        rate = rng.normal(blended_mean, std * (1 + 1-progress))
        failed = int(rng.integers(0, 3)) if progress < .5 else int(rng.choice([0, 0, 1]))
        destination = "Unknown host" if progress < .35 and rng.random() < .25 else rng.choice(allowed)

    return {
        "hour": hour,
        "packet_rate": max(0., float(rate)),
        "failed_logins": failed,
        "destination": str(destination)
      }

### Update system resources

In [39]:
def update_resources(cpu, memory, battery, state, rng):
    if state == "Attack-like":
        cpu += rng.uniform(4, 12)
        memory += rng.uniform(2, 7)
        battery -= rng.uniform(1.2, 2.2)
    elif state == "Suspicious":
        cpu += rng.uniform(-3, 7)
        memory += rng.uniform(-2, 4)
        battery -= rng.uniform(.5, 1.0)
    else:
        cpu += rng.uniform(-5, 3)
        memory += rng.uniform(-3, 2)
        battery -= rng.uniform(.1, .5)
    return (float(np.clip(cpu, 5, 100)),
            float(np.clip(memory, 10, 100)),
            float(np.clip(battery, 0, 100)))

### Calculate Security Accessment Score

In [40]:
def Behaviour(rate:float, mean:float, std:float, failed_logins:int ):
    z = abs(rate - mean) / max(std,1e-9)
    B = clip01(1 - z/5)
    B = clip01(B - min(failed_logins * .06, .30))
    return B

def Context(obs_hour:int, start_hours:int, end_hours:int, destination:str, allowed_destinations:list[str]):
    init = 1
    if not (start_hours <= obs_hour <= end_hours):
      init -= 0.4

    if (destination not in allowed_destinations):
      init -= 0.4
    return clip01(init)

def History(past_behaviours:list[float]):
    return float(np.mean(past_behaviours [-HISTORY_WINDOW:])) if past_behaviours else 1.0

def Risk(failed_logins:int, destination:str, allowed_destination:list[str], criticality:float):
    return clip01(0.3 * criticality + 0.3 * failed_logins + 0.4 * (destination not in allowed_destination))

def calculate_inputs(obs, device, past_behaviour, battery):
    b = Behaviour(obs["packet_rate"], device["mean"], device["std"], obs["failed_logins"])
    c = Context(obs["hour"],device["active"][0],device["active"][1],obs["destination"],device["allowed"])
    h = History(past_behaviour)
    r = Risk(obs["failed_logins"],obs["destination"],device["allowed"],device["criticality"])

    weights = BASE_WEIGHTS.copy()
    new_b_weight = weights["Behaviour"] * (.5 + .5 * battery / 100)
    removed = weights["Behaviour"] - new_b_weight

    other_total = weights["Context"] + weights["History"] + weights["Risk"]
    weights["Behaviour"] = new_b_weight

    for key in ("Context", "History", "Risk"):
        weights[key] += removed * BASE_WEIGHTS[key] / other_total

    S = (
        weights["Behaviour"] * b +
        weights["Context"] * c +
        weights["History"] * h +
        weights["Risk"] * (1-r)
      )

    return {
        "Behaviour": b,
        "Context": c,
        "History": h,
        "Risk": r,
        "Security_Assessment": clip01(S),
        "Effective_Weight_Behaviour": weights["Behaviour"]
      }

### Decide Action

In [41]:
def decide(trust):
    if trust >= .75:
        return "ALLOW"
    if trust >= .50:
        return "MONITOR"
    if trust >= .25:
        return "RESTRICT"
    return "ISOLATE"


### Run simulation of given alpha

In [42]:
def run_simulation(alpha=ALPHA, seed=42):
    rng = np.random.default_rng(seed)
    rows, events = [], []

    for device_id, device in DEVICES.items():
        trust = 1.0
        past_behaviour = []
        cpu, memory, battery = 20., 35., 100.
        previous_decision = None
        previous_trust = trust

        for t in range(N_STEPS):
            state = get_state(device, t)
            obs = generate_observation(device, t, state, rng)
            cpu, memory, battery = update_resources(cpu, memory, battery, state, rng)
            inputs = calculate_inputs(obs, device, past_behaviour, battery)
            S = inputs["Security_Assessment"]
            trust = clip01(alpha * trust + (1-alpha) * S)
            decision = decide(trust)

            row = {
                "Student_ID": STUDENT_ID,
                "Device_ID": device_id,
                "Device_Type": device["Device_Type"],
                "Time": t,
                "State": state,
                "Behaviour": inputs["Behaviour"],
                "Context": inputs["Context"],
                "History": inputs["History"],
                "Risk": inputs["Risk"],
                "Trust": trust,
                "Decision": decision,
                "CPU": cpu,
                "Memory": memory,
                "Battery": battery,
                "Detection_Delay": np.nan,
                "Recovery_Time": np.nan,
                "Packet_Rate": obs["packet_rate"],
                "Failed_Logins": obs["failed_logins"],
                "Destination": obs["destination"],
                "Security_Assessment": S,
                "Effective_Weight_Behaviour": inputs["Effective_Weight_Behaviour"],
            }
            rows.append(row)

            if previous_decision is not None and (decision != previous_decision or previous_trust - trust >= .15):
                events.append({
                    "Device_ID": device_id,
                    "Device_Type": device["Device_Type"],
                    "Time": t,
                    "State": state,
                    "Previous_Trust": previous_trust,
                    "New_Trust": trust,
                    "Behaviour": inputs["Behaviour"],
                    "Context": inputs["Context"],
                    "History": inputs["History"],
                    "Risk": inputs["Risk"],
                    "Decision": decision,
                    "Reason": "Decision changed" if decision != previous_decision else "Trust dropped by at least 0.15",
                })

            past_behaviour.append(inputs["Behaviour"])
            previous_decision, previous_trust = decision, trust

    df = pd.DataFrame(rows)

    for device_id, idx in df.groupby("Device_ID").groups.items():
        group = df.loc[idx]
        abnormal = group[group["State"].isin(["Suspicious", "Attack-like"])]
        recovery = group[group["State"] == "Recovery"]
        detection_delay = np.nan
        recovery_time = np.nan

        if not abnormal.empty:
            start = int(abnormal["Time"].min())
            flagged = abnormal[(abnormal["Time"] >= start) & abnormal["Decision"].isin(["RESTRICT", "ISOLATE"])]
            if not flagged.empty:
                detection_delay = int(flagged["Time"].min() - start)

        if not recovery.empty:
            start = int(recovery["Time"].min())
            recovered = recovery[(recovery["Time"] >= start) & (recovery["Trust"] >= .75)]
            if not recovered.empty:
                recovery_time = int(recovered["Time"].min() - start)

        df.loc[idx, "Detection_Delay"] = detection_delay
        df.loc[idx, "Recovery_Time"] = recovery_time

    return df, pd.DataFrame(events)

### Calculate metrics of runned simulation

In [43]:
def calculate_metrics(df):
    results = []
    for device_id, group in df.groupby("Device_ID", sort=False):
        abnormal = group[group["State"].isin(["Suspicious", "Attack-like"])]
        normal = group[group["State"] == "Normal"]
        recovery = group[group["State"] == "Recovery"]

        delay = group["Detection_Delay"].iloc[0]
        recovery_time = group["Recovery_Time"].iloc[0]
        false_rate = (normal["Decision"].isin(["RESTRICT", "ISOLATE"]).mean() if not normal.empty else np.nan)
        stability = normal["Trust"].std(ddof=0) if not normal.empty else np.nan
        results.append({
            "Device_ID": device_id,
            "Device_Type": group["Device_Type"].iloc[0],
            "Detection_Delay": delay,
            "False_Restriction_Rate": false_rate,
            "Recovery_Time": recovery_time,
            "Trust_Stability": stability,
            "Mean_Normal_Trust": normal["Trust"].mean() if not normal.empty else np.nan,
        })
    return pd.DataFrame(results)

### Comparing metrics of device across different alphas

In [44]:
def compare_alphas(alphas=(.3, .5, .7, .8)):
    tables = []
    for alpha in alphas:
        df, _ = run_simulation(alpha=alpha, seed=45)
        metrics = calculate_metrics(df)
        metrics["Alpha"] = alpha
        tables.append(metrics)
    return pd.concat(tables, ignore_index=True)

### plot result and observation graphs

In [45]:
def plot_required_graphs(df, device_id="D1"):
    data = df[df["Device_ID"] == device_id].sort_values("Time")

    if data.empty:
        raise ValueError(f"Unknown device {device_id}")

    # Graph 1: Trust vs time with states marked by shaded regions.
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(data["Time"], data["Trust"], label="Trust", linewidth=2)

    colors = {
        "Normal": "#d9ead3",
        "Suspicious": "#fff2cc",
        "Attack-like": "#f4cccc",
        "Recovery": "#cfe2f3"
    }

    for state in ["Normal", "Suspicious", "Attack-like", "Recovery"]:
        subset = data[data["State"] == state]
        if not subset.empty:
            ax.axvspan(
                subset["Time"].min(),
                subset["Time"].max()+1,
                alpha=.25,
                color=colors[state],
                label=state
            )

    ax.set(
        title=f"Graph 1: Trust Score vs Time — {device_id}",
        xlabel="Time step",
        ylabel="Trust score (0–1)"
    )

    ax.set_ylim(0, 1.05)
    ax.legend(ncol=3)
    ax.grid(alpha=.25)
    fig.tight_layout()
    fig.savefig(f"../images/graph1_trust_vs_time.png", dpi=160)
    plt.close(fig)

    # Graph 2: Behaviour vs time.
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(data["Time"], data["Behaviour"], label="Behaviour")
    ax.set(
        title=f"Graph 2: Behaviour vs Time — {device_id}",
        xlabel="Time step",
        ylabel="Behaviour score (0–1)"
    )
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=.25)
    fig.savefig(f"../images/graph2_behaviour_vs_time.png", dpi=160)
    plt.close(fig)

    # Graph 3: Four trust inputs together.
    fig, ax = plt.subplots(figsize=(12, 5))
    for col in ["Behaviour", "Context", "History", "Risk"]:
        ax.plot(data["Time"], data[col], label=col)

    ax.set(
        title=f"Graph 3: B, C, H and R vs Time — {device_id}",
        xlabel="Time step",
        ylabel="Input score (0-1)"
    )

    ax.set_ylim(0, 1.05)
    ax.legend(ncol=4)
    ax.grid(alpha=.25)
    fig.savefig(f"../images/graph3_trust_factors.png", dpi=160)
    plt.close(fig)

    # Graph 4: Trust and decision thresholds.
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(data["Time"], data["Trust"], label="Trust", linewidth=2)

    for y, label in [(.75, "ALLOW (0.75)"), (.50, "MONITOR (0.50)"),(.25, "RESTRICT (0.25)")]:
        ax.axhline(y, linestyle="--", label=label)

    ax.set(
        title=f"Graph 4: Trust and Decision Thresholds — {device_id}",
        xlabel="Time step", ylabel="Trust score (0-1)"
    )

    ax.set_ylim(0, 1.05)
    ax.legend(ncol=2)
    ax.grid(alpha=.25)
    fig.savefig(f"../images/graph4_trust_in_decision_threshold.png", dpi=160)
    plt.close(fig)

In [48]:
df, events = run_simulation(alpha=ALPHA, seed=42)
metrics = calculate_metrics(df)
alpha_results = compare_alphas()

required = [
    "Student_ID", "Device_ID", "Device_Type",
    "Time", "State",
    "Behaviour", "Context", "History", "Risk",
    "Trust", "Decision",
    "CPU", "Memory", "Battery",
    "Detection_Delay", "Recovery_Time",
]

df[required].to_csv("results.csv", index=False)
events.to_csv("security_events.csv", index=False)
metrics.to_csv("performance_metrics.csv", index=False)
alpha_results.to_csv("alpha_comparison.csv", index=False)

plot_required_graphs(df, "D1")

print("=== Zero-Trust IoT Simulation ===")
print(f"Devices: {df['Device_ID'].nunique()} | Steps/device: {N_STEPS} | Rows: {len(df)}")
print("\nDecision counts:\n", df["Decision"].value_counts().to_string())
print("\nPerformance metrics:\n", metrics.round(3).to_string(index=False))
print("\nAlpha comparison:\n", alpha_results.round(3).to_string(index=False),sep="\n")
print("\nSecurity events (first 10):\n", events.head(10).round(3).to_string(index=False))

=== Zero-Trust IoT Simulation ===
Devices: 5 | Steps/device: 120 | Rows: 600

Decision counts:
 Decision
ALLOW       352
MONITOR      99
RESTRICT     91
ISOLATE      58

Performance metrics:
 Device_ID                  Device_Type  Detection_Delay  False_Restriction_Rate  Recovery_Time  Trust_Stability  Mean_Normal_Trust
       D1                 Smart Camera              5.0                     0.0           20.0            0.030              0.884
       D2                 Smart Sensor              NaN                     0.0            NaN            0.020              0.889
       D3                   Smart Lock              8.0                     0.0           25.0            0.036              0.808
       D4                     Smart TV             19.0                     0.0           10.0            0.048              0.844
       D5 Medical/Environmental Sensor              4.0                     0.0           26.0            0.033              0.846

Alpha comparison:

De